# 17 — Multi-Tier GroupBy: Military Regiments & Score Analytics
> **Interview Prep & Technical Mastery Guide**
> 
> *A comprehensive, battle-tested reference for Python Data Science, Hierarchical Summaries, and Evaluation Metrics Interviews.*

---

## 📌 Executive Summary & Interview Expectations
This lab focuses on **hierarchical cohort analysis** across multiple categorical dimensions (`['regiment', 'company']`). In technical interviews, interviewers evaluate whether you can aggregate across multi-tiered organizations, unstack multi-level groupbys into clean matrices, and iterate over group objects idiomatically.

### Core Competencies Tested in this Module:
1. **Multi-Tier GroupBy**: Grouping by `['regiment', 'company']` and computing targeted statistics.
2. **Matrix Reshaping via `.unstack()`**: Converting hierarchical group aggregations into clean 2D comparison grids.
3. **Numeric Reduction Safety**: Using `numeric_only=True` to prevent runtime crashes on string columns.
4. **Group Iteration**: Idiomatic tuple unpacking `for name, group_df in df.groupby(...)` for custom batch operations.
5. **Interview Corner**: The unstacking mental model, targeted slicing vs full-table reductions, and score progression analysis.

## 1. Environment Setup & Data Construction

In [1]:
import numpy as np
import pandas as pd

raw_data = {
    "regiment": ["Nighthawks", "Nighthawks", "Nighthawks", "Nighthawks", "Dragoons", "Dragoons", "Dragoons", "Dragoons", "Scouts", "Scouts", "Scouts", "Scouts"],
    "company": ["1st", "1st", "2nd", "2nd", "1st", "1st", "2nd", "2nd", "1st", "1st", "2nd", "2nd"],
    "name": ["Miller", "Jacobson", "Ali", "Milner", "Cooze", "Jacon", "Ryaner", "Sone", "Sloan", "Parcher", "Blake", "Roache"],
    "preTestScore": [4, 24, 31, 2, 3, 4, 24, 31, 2, 3, 2, 3],
    "postTestScore": [25, 94, 57, 62, 70, 25, 94, 57, 62, 70, 62, 70]
}

regiment = pd.DataFrame(raw_data)
print("Regiment DataFrame constructed. Shape:", regiment.shape)
regiment.head(4)

Regiment DataFrame constructed. Shape: (12, 5)


,regiment,company,name,preTestScore,postTestScore
0,Nighthawks,1st,Miller,4,25
1,Nighthawks,1st,Jacobson,24,94
2,Nighthawks,2nd,Ali,31,57
3,Nighthawks,2nd,Milner,2,62


## 2. Targeted Group Aggregations

### ⚠️ Top Interview Anti-Pattern: Computing Means on String Columns
- In older Pandas: `regiment.groupby("regiment").mean()` silently dropped the `'name'` column.
- In **Pandas 2.x and 3.0**: Calling `.mean()` without `numeric_only=True` on tables with strings raises:
  `TypeError: Could not convert ['Miller', ...] to numeric`.
- **Targeted Best Practice**: Restrict to `['preTestScore']` directly before aggregating!

In [2]:
# Mean preTestScore for the 'Nighthawks' regiment
nighthawks_pretest = (
    regiment.groupby("regiment")["preTestScore"]
    .mean()
    .loc["Nighthawks"]
)
print(f"Mean preTestScore for Nighthawks: {nighthawks_pretest:.2f}")

Mean preTestScore for Nighthawks: 15.25


In [3]:
# General distribution statistics per company
regiment.groupby("company")["preTestScore"].describe()

,count,mean,std,min,25%,50%,75%,max
company,,,,,,,,
1st,6.0,6.666667,8.524475,2.0,3.00,3.5,4.00,24.0
2nd,6.0,15.500000,14.652645,2.0,2.25,13.5,29.25,31.0


In [4]:
# Mean preTestScore per company
regiment.groupby("company")["preTestScore"].mean()

company
1st     6.666667
2nd    15.500000
Name: preTestScore, dtype: float64

## 3. Multi-Level GroupBy & Reshaping via `.unstack()`

In [5]:
# Mean preTestScores grouped by regiment AND company (Hierarchical Series)
multi_group_series = regiment.groupby(["regiment", "company"])["preTestScore"].mean()
display(multi_group_series)

regiment    company
Dragoons    1st         3.5
            2nd        27.5
Nighthawks  1st        14.0
            2nd        16.5
Scouts      1st         2.5
            2nd         2.5
Name: preTestScore, dtype: float64

In [6]:
# Unstack: transforms multi-level series into a clean 2D comparison matrix
score_matrix = multi_group_series.unstack()
print("Score Matrix (Regiment x Company):")
display(score_matrix)

Score Matrix (Regiment x Company):


company,1st,2nd
regiment,,
Dragoons,3.5,27.5
Nighthawks,14.0,16.5
Scouts,2.5,2.5


## 4. Group Size and Full Numeric Reductions

In [7]:
# Observations per regiment and company cohort
regiment.groupby(["regiment", "company"]).size()

regiment    company
Dragoons    1st        2
            2nd        2
Nighthawks  1st        2
            2nd        2
Scouts      1st        2
            2nd        2
dtype: int64

In [8]:
# Full numeric reductions across all test scores
regiment.groupby(["regiment", "company"]).mean(numeric_only=True)

preTestScore  postTestScore
regiment   company                             
Dragoons   1st               3.5           47.5
           2nd              27.5           75.5
Nighthawks 1st              14.0           59.5
           2nd              16.5           59.5
Scouts     1st               2.5           66.0
           2nd               2.5           66.0

## 5. Idiomatic Iteration over GroupBy Objects

### 💡 Interview Note: The `(name, group_df)` Tuple Pattern
When custom non-vectorized operations are required (e.g. exporting separate files per regiment or running separate models), iterate using tuple unpacking:
```python
for name, group_df in df.groupby('col'):
    # name is the group key
    # group_df is a complete DataFrame subset
```

In [9]:
# Iterating over regiments
for reg_name, reg_group in regiment.groupby("regiment"):
    print(f"\n--- Regiment: {reg_name} (Total Personnel: {len(reg_group)}) ---")
    display(reg_group[["company", "name", "preTestScore", "postTestScore"]].head(2))


--- Regiment: Dragoons (Total Personnel: 4) ---


,company,name,preTestScore,postTestScore
4,1st,Cooze,3,70
5,1st,Jacon,4,25



--- Regiment: Nighthawks (Total Personnel: 4) ---


,company,name,preTestScore,postTestScore
0,1st,Miller,4,25
1,1st,Jacobson,24,94



--- Regiment: Scouts (Total Personnel: 4) ---


,company,name,preTestScore,postTestScore
8,1st,Sloan,2,62
9,1st,Parcher,3,70


## 6. Multi-Tier GroupBy Cheat Sheet

| Task | Idiomatic Syntax | Key Benefit |
| :--- | :--- | :--- |
| **Multi-Tier Group** | `df.groupby(['A', 'B'])['C'].mean()` | Creates MultiIndex series |
| **Reshape to Grid** | `df.groupby(['A', 'B'])['C'].mean().unstack()` | 2D matrix layout |
| **Cohort Counts** | `df.groupby(['A', 'B']).size()` | Fast observation count per pair |
| **Tuple Iteration** | `for name, sub_df in df.groupby('A'):` | Clean batch processing per group |

---
## 🎯 7. Technical Interview Corner: Tricky Questions & Drills

### Q1: When is iterating over a GroupBy object actually justified?
**Question**: Interviewers often warn that *"loops in Pandas are an anti-pattern."* When is iterating over a `groupby` object (`for name, group in df.groupby():`) legitimate and necessary in production?

**Answer**:
1. **Side Effects / I/O**: Exporting separate CSV/Parquet files per tenant/region (e.g. `group.to_csv(f'{name}.csv')`).
2. **External Model Fitting**: Fitting separate statistical or machine learning models (e.g. `LinearRegression().fit(group[X], group[y])`) per group.
3. **Complex Plotting**: Generating distinct customized Matplotlib figures per segment.
*For pure calculations (mean, sum, rank), vectorized `.agg()` or `.transform()` should ALWAYS be used.*

### Q2: Advanced Interview Challenge: Score Improvement Ratio
**Challenge**: Compute the percentage improvement from `preTestScore` to `postTestScore` for each individual soldier:
$$\text{Improvement} = \frac{\text{postTestScore} - \text{preTestScore}}{\text{preTestScore}} \times 100$$
Then, find the **company within each regiment** that achieved the highest median improvement!

In [10]:
# Solution to Coding Challenge
regiment_enhanced = regiment.assign(
    improvement_pct=lambda df: ((df["postTestScore"] - df["preTestScore"]) / df["preTestScore"]) * 100
)

best_companies = (
    regiment_enhanced.groupby(["regiment", "company"])["improvement_pct"]
    .median()
    .round(1)
    .unstack()
)

print("Median Improvement (%) by Regiment and Company:")
display(best_companies)

Median Improvement (%) by Regiment and Company:


company,1st,2nd
regiment,,
Dragoons,1379.2,187.8
Nighthawks,408.3,1541.9
Scouts,2616.7,2616.7
